# YOLO11m-pose — landing base @ imgsz 1280 (Kaggle)

> **Project authored by Claude (Anthropic, Opus 4.8)** for `a6_gaivotas_ccd2026`.

Trains **yolo11m-pose** (5 keypoints: center + 4 corners) at **imgsz 1280** for
*maximum quality* (teacher / prelabel model). Attach dataset
`yabrick/base-pouso-yolo-pose` (already YOLO-pose format). GPU + Internet **ON**.

Outputs in /kaggle/working: `best.pt`, `best_float32.tflite`, `best_float16.tflite`,
`best_ncnn_model/`. Staged dataset is deleted at the end to keep the download small.

In [ ]:
# Kaggle's base torch is 2.10+cu128, which DROPPED Pascal (sm_60) -> the assigned
# P100 GPU throws cudaErrorNoKernelImageForDevice. Pin torch to the last
# Pascal-compatible cu121 build so the P100 works (the kernels API can't pick a T4).
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics
import os
os.environ['WANDB_MODE'] = 'disabled'; os.environ['WANDB_DISABLED'] = 'true'
import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# fail fast if torch/GPU are still incompatible
_t = torch.zeros(8, device='cuda'); print('cuda op ok, sum =', float((_t + 1).sum()))

# ultralytics checkpoints are trusted; torch>=2.6 defaults weights_only=True
_orig_load = torch.load
def _patched_load(f, map_location=None, pickle_module=None, **kw):
    kw['weights_only'] = False
    if pickle_module is not None:
        return _orig_load(f, map_location=map_location, pickle_module=pickle_module, **kw)
    return _orig_load(f, map_location=map_location, **kw)
torch.load = _patched_load

In [ ]:
# Robustly locate the attached dataset (mount name can vary) and stage it into
# /kaggle/working (writable -> label caches). Print the input tree for diagnosis.
import os, glob, zipfile, shutil
from pathlib import Path

print('== /kaggle/input tree (depth<=3) ==')
for root, dirs, files in os.walk('/kaggle/input'):
    d = root.replace('/kaggle/input', '').count('/')
    if d <= 3:
        print('  ' * d, os.path.basename(root) or root, '| dirs:', dirs[:6], '| files:', files[:6])

DST = Path('/kaggle/working/dataset')
if DST.exists(): shutil.rmtree(DST)
DST.mkdir(parents=True)

# Case A: extracted layout — find images/train anywhere under /kaggle/input
tr = glob.glob('/kaggle/input/**/images/train', recursive=True)
if tr:
    root = Path(tr[0]).parents[1]
    print('found extracted dataset at', root)
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        shutil.copytree(root / sub, DST / sub)
else:
    # Case B: zipped layout (images.zip / labels.zip) — extract preserving structure
    iz = glob.glob('/kaggle/input/**/images.zip', recursive=True)
    lz = glob.glob('/kaggle/input/**/labels.zip', recursive=True)
    assert iz and lz, f'dataset not found (see tree above). images.zip={iz} labels.zip={lz}'
    print('extracting', iz[0], lz[0])
    for z, top in ((iz[0], 'images'), (lz[0], 'labels')):
        with zipfile.ZipFile(z) as zf:
            names = zf.namelist()
            # if archive root already includes the top dir, extract to DST; else into DST/top
            base = DST if any(n.startswith(top + '/') for n in names) else (DST / top)
            zf.extractall(base)

assert (DST / 'images/train').exists(), 'layout unexpected: ' + str([p.name for p in DST.rglob('*')][:20])
n_tr = len(list((DST / 'images/train').glob('*')))
n_va = len(list((DST / 'images/val').glob('*')))
print(f'staged: train {n_tr} | val {n_va}')

Path('/kaggle/working/data.yaml').write_text(
    'path: /kaggle/working/dataset\n'
    'train: images/train\n'
    'val: images/val\n'
    'kpt_shape: [5, 3]\n'
    'flip_idx: [0, 2, 1, 4, 3]\n'
    'names:\n  0: base_pouso\n')
print(Path('/kaggle/working/data.yaml').read_text())

In [ ]:
import os
os.chdir('/kaggle/working')
from ultralytics import YOLO
from pathlib import Path

model = YOLO('yolo11n-pose.pt')   # COCO-pretrained NANO -> light final model for RPi
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=300, imgsz=320, batch=64, device=0,   # nano @320: light final model for RPi
    project='base_pouso', name='n320', patience=60,
    optimizer='auto', cos_lr=True, close_mosaic=25,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.0, mosaic=1.0,
)
RUN_DIR = Path(model.trainer.save_dir)
print('RUN_DIR:', RUN_DIR)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(RUN_DIR / 'results.csv'); df.columns = df.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df['epoch'], df['metrics/mAP50(P)'],    label='mAP50 (pose)')
ax[0].plot(df['epoch'], df['metrics/mAP50-95(P)'], label='mAP50-95 (pose)')
ax[0].set_title('mAP - Pose'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(df['epoch'], df['train/pose_loss'], label='train')
ax[1].plot(df['epoch'], df['val/pose_loss'],   label='val')
ax[1].set_title('Pose loss'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()
print('Best mAP50(P):    %.4f' % df['metrics/mAP50(P)'].max())
print('Best mAP50-95(P): %.4f' % df['metrics/mAP50-95(P)'].max())

In [ ]:
# Export best.pt. Each format is isolated so one failing (e.g. tflite needs a
# newer torch than the P100 pin allows) doesn't abort the kernel or the cleanup.
from ultralytics import YOLO
best = RUN_DIR / 'weights' / 'best.pt'
print('best.pt: %.1f MB' % (best.stat().st_size / 1e6))
m = YOLO(best)
for fmt in ('ncnn', 'tflite'):        # ncnn first (the deployment target)
    try:
        print(fmt, '->', m.export(format=fmt, imgsz=320))
    except Exception as e:
        print(fmt, 'FAILED:', repr(e))

In [ ]:
import shutil
from pathlib import Path
W = Path('/kaggle/working'); wdir = RUN_DIR / 'weights'
shutil.copy(wdir / 'best.pt', W / 'best.pt')
for f in wdir.glob('best*.tflite'):
    shutil.copy(f, W / f.name)
nc = wdir / 'best_ncnn_model'
if nc.exists():
    dst = W / 'best_ncnn_model'
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(nc, dst)
if (W / 'dataset').exists(): shutil.rmtree(W / 'dataset')  # shrink output
print('deliverables:')
for p in sorted(W.iterdir()):
    sz = sum(f.stat().st_size for f in p.rglob('*')) if p.is_dir() else p.stat().st_size
    print(f'  {p.name:28s} {sz/1e6:8.1f} MB')